# MuscleMimic: mjlab GPU Backend

Colab + CUDA (T4): [Open in Colab](https://colab.research.google.com/drive/144wHsu_UBVofZqXRTOUWY33ZscziA76R).

Demonstrates running the MuscleMimic fullbody task on the **mjlab** backend,
which uses [Warp](https://github.com/NVIDIA/warp) for GPU-parallel simulation
(falls back to CPU when CUDA is unavailable).

Steps covered:

1. Verify mjlab / mujoco_warp / warp availability
2. Load a motion clip and register the mjlab task
3. Create a `ManagerBasedRlEnv` and run a zero-action rollout
4. Wrap with `RslRlVecEnvWrapper` and run PPO via `MjlabOnPolicyRunner`
5. Architecture summary


## 0. Paths

Automated runs (e.g. `jupyter nbconvert --execute`) may start the kernel with a working directory that is **not** inside this repo (often `/tmp`). In that case set **`MYOSUITE_REPO`** to the repository root (the directory that contains the `myosuite/` package). Optional smoke training length: **`MJLAB_MIMIC_DEMO_ITERS`** (defaults to `500`). Optional scale knobs: **`MJLAB_MIMIC_NUM_ENVS`** (defaults to `16`), **`MJLAB_MIMIC_STEPS_PER_ENV`** (defaults to `64`), **`MJLAB_MIMIC_RENDER_FRAMES`** (defaults to `300`), **`MJLAB_MIMIC_NCONMAX`**, and **`MJLAB_MIMIC_NJMAX`**. Optional logging backend: **`MJLAB_MIMIC_LOGGER`** (defaults to `tensorboard`; set to `wandb` only when authenticated).


In [ ]:
import os
import sys
from pathlib import Path


def _find_myosuite_repo() -> Path:
    """Return repo root (directory that contains ``myosuite/``)."""
    env = os.environ.get('MYOSUITE_REPO', '').strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / 'myosuite').is_dir():
            return p
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'myosuite').is_dir():
            return d
    raise RuntimeError(
        'Cannot find myosuite repo root. Run the notebook from inside the clone, '
        'or set MYOSUITE_REPO to the repository root (parent of the myosuite/ package).'
    )


_repo_root = _find_myosuite_repo()

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
if str(_repo_root / 'tutorials') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'tutorials'))

print('Repo root:', _repo_root)


## 1. Verify mjlab / mujoco_warp / warp / musclemimic

`mjlab` requires:
- `warp-lang` — GPU kernel framework (CPU fallback when no CUDA)
- `mujoco_warp` — MuJoCo physics compiled to Warp kernels
- `mjlab` — task / env / runner framework built on the above
- `musclemimic` + dependencies — MuscleMimic package (easiest to install via myosuite[musclemimic])

If missing, run the following two lines, where "cu130" should be replaced by the CUDA version supported by your system (the latest version can be found by running `nvidia-smi` in the terminal), e.g., cu126 for CUDA 12.6:
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130 --force-reinstall
pip install myosuite[warp] myosuite[musclemimic]
```




In [ ]:
import warnings
warnings.filterwarnings('ignore')

_MJLAB_OK = False
_MJLAB_RUN = False
_DEVICE = 'cpu'
CLIP_PATH = None

try:
    import warp as wp
    import mujoco_warp
    import mjlab
    import platform
    devices = wp.get_devices()
    _DEVICE = 'cuda' if any(d.is_cuda for d in devices) else 'cpu'
    _MJLAB_OK = True
    _MJLAB_RUN = _DEVICE == 'cuda' and platform.system() != 'Darwin'
    # Register MyoSuite task ids with mjlab (entry point is not always run on ``import mjlab``).
    try:
        import importlib
        importlib.import_module('myosuite.envs.myo.backends.mjlab')
    except ImportError:
        pass
    print(f'warp {wp.__version__}  |  mujoco_warp {mujoco_warp.__version__}  |  mjlab OK')
    print(f'Warp devices: {[str(d) for d in devices]}')
    print(f'Using device: {_DEVICE}')
    if not _MJLAB_RUN:
        print('[SKIP] mjlab env/rollout cells need Linux + CUDA (Warp can hang on CPU/macOS).')
except ImportError as e:
    print(f'[SKIP] mjlab not available: {e}')
    print('Install with: pip install myosuite[mjlab] myosuite[musclemimic]')


## 2. Load the motion clip

The mjlab mimic task requires a retargeted `.npz` clip with
`xpos`, `xquat`, `qpos`, `qvel`, and `site_xpos` arrays.
Use `hf_hub_download(...)` to resolve the local file path in a machine-agnostic way:

```python
from pathlib import Path
from huggingface_hub import hf_hub_download

CLIP_REPO_ID = 'amathislab/musclemimic-retargeted'
CLIP_FILENAME = 'MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz'

CLIP_PATH = Path(
    hf_hub_download(
        repo_id=CLIP_REPO_ID,
        filename=CLIP_FILENAME,
        repo_type='dataset',
    )
)
```

**NOTE:** You may need to first go to https://huggingface.co/datasets/amathislab/musclemimic-retargeted and accept the licence requirements for this dataset; afterwards, create an huggingface access token and paste it into the next cell:


In [ ]:
import os
os.environ["HF_TOKEN"] = ""  #access token from your huggingface account goes here

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

CLIP_REPO_ID = 'amathislab/musclemimic-retargeted'
CLIP_FILENAME = 'MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz'
CLIP_PATH = None
try:
    CLIP_PATH = Path(
        hf_hub_download(
            repo_id=CLIP_REPO_ID,
            filename=CLIP_FILENAME,
            repo_type='dataset',
        )
    )
    if not CLIP_PATH.is_file():
        raise FileNotFoundError(CLIP_PATH)
    print('Clip:', CLIP_PATH)
except Exception as e:
    CLIP_PATH = None
    print(f'[SKIP] Full-body clip not downloaded ({type(e).__name__}: {e})')
    print('Need network access to huggingface.co/datasets/amathislab/musclemimic-retargeted')


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

BIMANUAL_CLIP_PATH = None
try:
    BIMANUAL_CLIP_PATH = Path(
        hf_hub_download(
            repo_id='amathislab/musclemimic-bimanual-retargeted',
            filename='MyoBimanualArm/gmr/BioMotionLab_NTroje/rub001/0011_lifting_light1_poses.npz',
            repo_type='dataset',
        )
    )
    print('Optional bimanual clip:', BIMANUAL_CLIP_PATH)
except Exception as e:
    print(
        '[SKIP] Bimanual clips are on a gated Hugging Face dataset '
        f'({type(e).__name__}). Request access at '
        'https://huggingface.co/datasets/amathislab/musclemimic-bimanual-retargeted'
    )
    print('The full-body CLIP_PATH from the previous cell is unchanged.')


## 3. Load MotionClip and register task

`load_motion_clip` validates array shapes and populates a typed
`MotionClip` dataclass.  `register_mimic_mjlab_tasks_with_clip` creates
a `ManagerBasedRlEnvCfg` for `myoMimicFullbody-v0` using this clip as the
trajectory target source.


In [ ]:
if _MJLAB_OK and _MJLAB_RUN and CLIP_PATH:
    import numpy as np
    from mjlab.tasks.registry import register_mjlab_task
    from myosuite.core.trajectory_io import load_motion_clip
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        default_mimic_clip_on_policy_runner_cfg,
        register_mimic_mjlab_tasks_with_clip,
    )

    clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
    print(f'Clip loaded: T={clip.qpos.shape[0]} frames  nq={clip.qpos.shape[1]}')

    register_mimic_mjlab_tasks_with_clip(
        register_mjlab_task=register_mjlab_task,
        rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
        clip=clip,
        use_lookahead=True,
    )
    print('Registered: myoMimicFullbody-v0')


## 4. Create `ManagerBasedRlEnv` and run a rollout

Instantiate the mjlab env with 2 parallel environments and step it
for 10 steps with zero actions to verify physics runs correctly.

The cell registers `myoMimicFullbody-v0` if it is not already in the
registry (e.g. after a kernel restart or when running cells out of order),
then calls `load_env_cfg` to retrieve the config.

In [ ]:
if _MJLAB_OK and _MJLAB_RUN and CLIP_PATH:
    import torch
    from mjlab.tasks.registry import list_tasks, load_env_cfg, register_mjlab_task
    from mjlab.envs import ManagerBasedRlEnv

    # Register the task if cell 8 was skipped (e.g. kernel restart in Colab).
    if 'myoMimicFullbody-v0' not in list_tasks():
        from myosuite.core.trajectory_io import load_motion_clip
        from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
            default_mimic_clip_on_policy_runner_cfg,
            register_mimic_mjlab_tasks_with_clip,
        )
        _clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
        register_mimic_mjlab_tasks_with_clip(
            register_mjlab_task=register_mjlab_task,
            rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
            clip=_clip,
            use_lookahead=True,
        )
        print('Registered: myoMimicFullbody-v0')

    NUM_ENVS = int(os.environ.get('MJLAB_MIMIC_NUM_ENVS', '16'))  # increase to 512-2048 on GPU
    env_cfg = load_env_cfg('myoMimicFullbody-v0')
    env_cfg.scene.num_envs = NUM_ENVS
    if os.environ.get('MJLAB_MIMIC_NCONMAX'):
        env_cfg.sim.nconmax = int(os.environ['MJLAB_MIMIC_NCONMAX'])
    if os.environ.get('MJLAB_MIMIC_NJMAX'):
        env_cfg.sim.njmax = int(os.environ['MJLAB_MIMIC_NJMAX'])

    train_env = ManagerBasedRlEnv(cfg=env_cfg, device=_DEVICE)
    obs, _ = train_env.reset()
    act_dim = train_env.action_space.shape[-1]
    obs_dim = obs['actor'].shape[-1]
    print(f'obs_dim={obs_dim}  act_dim={act_dim}  num_envs={NUM_ENVS}')

    for step in range(10):
        actions = torch.zeros(NUM_ENVS, act_dim, device=_DEVICE)
        obs, rew, term, trunc, info = train_env.step(actions)

    print(f'Rollout OK — obs[actor] shape: {obs["actor"].shape}')


## 5. PPO training with `MjlabOnPolicyRunner`

`RslRlVecEnvWrapper` adapts the `ManagerBasedRlEnv` to the `rsl_rl` API.
`MjlabOnPolicyRunner` runs the PPO update loop.

**Why raw `RslRlOnPolicyRunnerCfg()` can plateau (e.g. mean return ~10–15)** while
`tutorials/5.2_files/train_mimic.py` on the same clip keeps improving:

| Aspect | `train_mimic` + `MuscleMimicClipEnvV0` | mjlab default runner cfg |
|--------|----------------------------------------|---------------------------|
| Observations | `RunningMeanStd` when `use_obs_normalizer=True` (default) | `obs_normalization=False` on actor/critic |
| MLP | 256-wide × 4 layers, SiLU + LayerNorm | 128³, ELU |
| LR / entropy | 3e-4, entropy 1e-3 | 1e-3, 5e-3 |
| LR schedule | Linear anneal | `adaptive` + `desired_kl` (can shrink LR early) |
| Actions | Direct muscle `ctrl` in `[0, 1]` (Gaussian + clip) | `[-1,1]` → `sigmoid(5*(a-0.5))` into muscles |

The registration cell above uses ``default_mimic_clip_on_policy_runner_cfg`` from
``mimic_mjlab_env`` so the task’s stored RL config matches that regime.  The
training cell builds the runner from the same helper and calls
``install_episode_reward_logging_patch()`` so **mean episode return** in logs
matches rsl_rl’s internal buffers (upstream rsl_rl only fills ``rewbuffer``
when a SummaryWriter exists).

You still need **many more steps** than the CPU demo for gym-class convergence
(order 10⁷–10⁹ env steps).  Optional: ``python scripts/bench_mjlab_mimic_ppo.py``
compares baseline vs tuned configs headlessly.

**Hardware** (adjust to your machine):

| Setting | CPU (demo) | GPU (training) |
|---------|-----------|----------------|
| `NUM_ENVS` | 16 | 512–2048 |
| `num_steps_per_env` | 64 | 128 |
| `max_iterations` | 500 | 5 000+ |

On CPU, 500 × 16 × 64 ≈ **512 k** env steps — a smoke test only.


In [ ]:
if _MJLAB_OK and _MJLAB_RUN and CLIP_PATH:
    import dataclasses
    import os
    import torch
    from mjlab.rl import RslRlVecEnvWrapper, MjlabOnPolicyRunner

    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        default_mimic_clip_on_policy_runner_cfg,
    )
    from myosuite.envs.myo.backends.mjlab.rsl_rl_logger_episode_patch import (
        install_episode_reward_logging_patch,
    )

    TRAIN_LOG_DIR = '/tmp/mjlab_mimic'
    # Override for CI / nbconvert smoke: ``MJLAB_MIMIC_DEMO_ITERS=2 jupyter nbconvert ...``
    NUM_ITERS = int(os.environ.get('MJLAB_MIMIC_DEMO_ITERS', '500'))
    STEPS_PER_ENV = int(os.environ.get('MJLAB_MIMIC_STEPS_PER_ENV', '64'))

    wrapped = RslRlVecEnvWrapper(train_env)

    # Same defaults as registration (default_mimic_clip_on_policy_runner_cfg).
    runner_cfg = dataclasses.asdict(default_mimic_clip_on_policy_runner_cfg())
    runner_cfg['logger'] = os.environ.get('MJLAB_MIMIC_LOGGER', 'tensorboard')
    runner_cfg['num_steps_per_env'] = STEPS_PER_ENV
    runner_cfg['max_iterations'] = NUM_ITERS
    # Save at least once before the final iter so short smoke runs still get a ckpt.
    runner_cfg['save_interval'] = min(100, max(1, NUM_ITERS))

    install_episode_reward_logging_patch()

    runner = MjlabOnPolicyRunner(
        env=wrapped,
        train_cfg=runner_cfg,
        log_dir=TRAIN_LOG_DIR,
        device=_DEVICE,
    )
    runner.learn(num_learning_iterations=NUM_ITERS, init_at_random_ep_len=True)
    print(f'Training complete — checkpoints in {TRAIN_LOG_DIR}')


## 6. Render trained policy as inline video

Loads the latest checkpoint from the training log directory, runs it in a
single-env `ManagerBasedRlEnv` with `render_mode="rgb_array"`, and writes
an MP4.  Falls back to a zero-action rollout if no checkpoint is found.


In [ ]:
if _MJLAB_OK and _MJLAB_RUN and CLIP_PATH:
    import glob, torch, mediapy
    from pathlib import Path
    from IPython.display import Video, display
    from mjlab.envs import ManagerBasedRlEnv
    from mjlab.viewer import ViewerConfig

    from mjlab.tasks.registry import list_tasks, load_env_cfg, register_mjlab_task
    from myosuite.envs.myo.backends.mjlab.mimic_mjlab_env import (
        default_mimic_clip_on_policy_runner_cfg,
        register_mimic_mjlab_tasks_with_clip,
    )

    N_FRAMES   = int(os.environ.get('MJLAB_MIMIC_RENDER_FRAMES', '300'))
    FPS        = 30
    VIDEO_PATH = '/tmp/mjlab_policy_rollout.mp4'

    # ── load latest checkpoint (sort by iteration — glob order is lexical) ─
    import re

    def _ckpt_iter(path: str) -> int:
        m = re.search(r'model_(\d+)\.pt$', path.replace('\\', '/'))
        return int(m.group(1)) if m else -1

    ckpts = sorted(
        glob.glob(f'{TRAIN_LOG_DIR}/**/model_*.pt', recursive=True),
        key=_ckpt_iter,
    )
    policy_fn = None
    if ckpts:
        latest = ckpts[-1]
        print(f'Loading checkpoint: {latest}')
        # rsl_rl EmpiricalNormalization buffers can be inference tensors after
        # training; load_state_dict must run under inference_mode (PyTorch 2.x).
        with torch.inference_mode():
            runner.load(latest, map_location=_DEVICE)
        policy_fn = runner.get_inference_policy(device=_DEVICE)
    else:
        print('No checkpoint found — rendering zero-action rollout.')

    # ── render env ────────────────────────────────────────────────────────
    if 'myoMimicFullbody-v0' not in list_tasks():
        from myosuite.core.trajectory_io import load_motion_clip
        _clip = load_motion_clip(CLIP_PATH, expected_nq=89, expected_nv=88)
        register_mimic_mjlab_tasks_with_clip(
            register_mjlab_task=register_mjlab_task,
            rl_cfg_fn=default_mimic_clip_on_policy_runner_cfg,
            clip=_clip,
            use_lookahead=True,
        )
    render_cfg = load_env_cfg('myoMimicFullbody-v0', play=True)
    render_cfg.scene.num_envs = 1
    render_cfg.viewer = ViewerConfig(width=640, height=360, distance=3.5, elevation=-20.0)

    render_env = ManagerBasedRlEnv(cfg=render_cfg, device=_DEVICE, render_mode='rgb_array')
    obs, _ = render_env.reset()
    act_dim_r = render_env.action_space.shape[-1]

    frames = []
    for _ in range(N_FRAMES):
        if policy_fn is not None:
            with torch.no_grad():
                # rsl_rl MLPModel expects a TensorDict with keys from obs_groups (e.g. 'actor').
                actions = policy_fn(obs)
        else:
            actions = torch.zeros(1, act_dim_r, device=_DEVICE)
        obs, _, _, _, _ = render_env.step(actions)
        frame = render_env.render()
        if frame is not None:
            frames.append(frame)

    render_env.close()
    if not frames:
        raise RuntimeError('No frames captured; check render_mode and viewer config.')
    print(f'Captured {len(frames)} frames  ({frames[0].shape} each)')

    mediapy.write_video(VIDEO_PATH, frames, fps=FPS)
    display(Video(VIDEO_PATH, embed=True, width=640, height=360))


## 7. Architecture summary

```
motion clip (.npz)
  └── load_motion_clip()  →  MotionClip
        └── register_mimic_mjlab_tasks_with_clip()
              └── ManagerBasedRlEnvCfg
                    ├── SceneCfg  (EntityCfg → MjSpec → mujoco_warp / Warp)
                    ├── ObservationManager  (groups: policy / actor / critic)
                    ├── ActionManager  (MyoMuscleActivationAction)
                    ├── RewardManager  (DeepMimic composite)
                    └── TerminationManager  (time_out + mimic_deviation)

ManagerBasedRlEnv
  └── RslRlVecEnvWrapper
        └── MjlabOnPolicyRunner  →  PPO (rsl_rl)
```

**Key files**

| File | Role |
|------|------|
| `myosuite/envs/myo/backends/mjlab/mimic_mjlab_env.py` | Task wiring, obs/reward/termination closures, init state |
| `myosuite/envs/myo/backends/mjlab/register_mjlab_tasks.py` | `MyoMuscleActivationAction`, actuator helpers |
| `myosuite/core/trajectory_io.py` | `load_motion_clip`, `MotionClip` dataclass |
| `myosuite/envs/myo/backends/mjlab/clip_trajectory_source.py` | Per-env frame sampling from clip |
